# 01. Project Overview

## Thyroid Nodule Classification on TN5000 — VGG16T Baseline, Fine-Tuning and ResNet50

This notebook provides the code used for a reproducible comparison of single-CNN models for benign vs. malignant thyroid-nodule ultrasound classification on the public **TN5000** dataset.

### Reference paper
Zhu et al. (2021), *Thyroid ultrasound image classification using a convolutional neural network*.

The VGG16T result reported in that paper (86.43% internal accuracy) was obtained on the authors' own cohort. **The experiments in this notebook use the separate public TN5000 dataset and therefore are not a direct reproduction of that result.**

### Experimental design
- **Dataset:** TN5000, with the official image-level **7:1:2** split (3,500 train / 500 validation / 1,000 test).
- **Preprocessing:** expert bounding-box ROI crop with a 15% context margin → resize to 224×224 → Keras VGG16 `preprocess_input`.
- **Augmentation:** training-only horizontal flip, small rotation/zoom/translation and contrast adjustment.
- **Class imbalance:** handled using balanced class weights.
- **Models:**
  1. **VGG16T baseline** — frozen ImageNet VGG-16 backbone + 2×(Dense 4096 + BatchNorm + ReLU + Dropout) + sigmoid output.
  2. **VGG16T fine-tuned** — baseline checkpoint with VGG-16 block 5 unfrozen while BatchNorm remains frozen.
  3. **ResNet50** — frozen ImageNet backbone + global average pooling + Dense 256 + Dropout + sigmoid output.
- **Evaluation:** accuracy, precision, sensitivity, specificity, F1, AUC, confusion matrices and ROC curves.
- **Additional analyses:** validation-only threshold selection, bootstrap 95% confidence intervals, Grad-CAM, parameter count, checkpoint size and CPU inference speed.

### Public-release notes
- The notebook contains **no personal names, email addresses, private directories, local output directory mounts, API keys, passwords, tokens or credentials**.
- The TN5000 data are downloaded from the public dataset release; the dataset itself is **not bundled** with this notebook.
- All generated files are written to a local `results/` directory.
- Model checkpoints are written to `results/checkpoints/`.
- The training sections can be skipped if compatible checkpoints are already available locally.
- **Do not upload private credentials or personal files into this notebook.**
- The official split is required. If the split manifests cannot be found, the notebook stops rather than creating a new random split.

### Contents
| # | Section |
|---|---------|
| 01 | Project Overview |
| 02 | Environment & Imports |
| 03 | TN5000 Dataset Download |
| 04 | Dataset Extraction |
| 05 | Annotation Parsing & Official Train/Validation/Test Split |
| 06 | ROI Preprocessing & Input Pipeline |
| 07 | VGG16T Baseline |
| 08 | VGG16T Fine-Tuning |
| 09 | ResNet50 |
| 10 | Model Comparison |
| 11 | ROC Curves |
| 12 | Threshold Analysis |
| 13 | Bootstrap 95% Confidence Intervals |
| 14 | Grad-CAM Explainability |
| 15 | Model Parameters & Efficiency |
| 16 | Final Results Tables & Figures |
| 17 | Research Conclusions |

### Reproducibility
The notebook uses a fixed random seed (`42`) for the data pipeline and training setup. The reported results in Section 17 are the results from the completed research run. Re-running the training cells may produce small differences because training is stochastic and hardware/software environments can differ.


# 02. Environment & Imports

Install the required packages, centralise imports, set the random seed, and create portable local output folders.

This notebook is designed for Google Colab or a local Python/Jupyter environment. A GPU is recommended for training, but the evaluation and analysis sections can run on CPU when compatible checkpoints are available.


In [ ]:
!pip -q install scikit-learn seaborn lxml

In [ ]:
import os, json, gc, time, zipfile, tarfile, random, shutil, requests
import xml.etree.ElementTree as ET
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import cv2
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.applications.vgg16 import preprocess_input
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix, classification_report)

SEED=42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow:",tf.__version__,"GPU:",tf.config.list_physical_devices("GPU"))

### 02.1 Local output folders

All generated checkpoints, tables, figures and prediction files are stored under the local `results/` directory. No local output directory connection is required.


In [ ]:
# Portable local workspace
WORK_ROOT = Path.cwd() / "TN5000"
OUTPUT_ROOT = Path.cwd() / "results"

CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
FIG_DIR = OUTPUT_ROOT / "figures"
TABLE_DIR = OUTPUT_ROOT / "tables"
PRED_DIR = OUTPUT_ROOT / "predictions"

for d in [CHECKPOINT_DIR, FIG_DIR, TABLE_DIR, PRED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Workspace:", WORK_ROOT)
print("Results:", OUTPUT_ROOT)


In [ ]:
# Main dataset directories
ROOT = WORK_ROOT
RAW = ROOT / "raw"
PROC = ROOT / "processed"

RAW.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)

print("Dataset root:", ROOT)
print("Raw data:", RAW)
print("Processed data:", PROC)


# 03. TN5000 Dataset Download
The dataset is public on Figshare. The code queries the Figshare API and downloads the files automatically — no manual upload.

In [ ]:
# Public TN5000 release
# The dataset is downloaded automatically from the Figshare API.
# No private account or credential is required.

ARTICLE_ID = 28455641

meta = requests.get(
    f"https://api.figshare.com/v2/articles/{ARTICLE_ID}",
    timeout=60
)
meta.raise_for_status()
article = meta.json()

print(article["title"])
for f in article["files"]:
    print(f"{f['name']} — {f['size']/1e6:.2f} MB")


In [ ]:
def download(url,dst):
    if dst.exists() and dst.stat().st_size>0: return
    with requests.get(url,stream=True,timeout=120) as r:
        r.raise_for_status()
        with open(dst,"wb") as out:
            for c in r.iter_content(1024*1024):
                if c: out.write(c)
for f in article["files"]:
    p=RAW/f["name"]; download(f["download_url"],p); print("Downloaded:",p)

# 04. Dataset Extraction
Unzips the downloaded archives. The data used here lives in `TN5000_forReview/` (`JPEGImages/`, `Annotations/`, `ImageSets/`).

In [ ]:
for p in RAW.iterdir():
    try:
        if p.suffix.lower()==".zip":
            with zipfile.ZipFile(p) as z: z.extractall(RAW)
        elif p.name.lower().endswith((".tar",".tar.gz",".tgz")):
            with tarfile.open(p,"r:*") as t: t.extractall(RAW)
    except Exception as e: print("extract:",p,e)
print("\nFolders:")
for p in sorted(RAW.rglob("*")):
    if p.is_dir() and len(p.relative_to(RAW).parts)<=2: print(p)

# 05. Annotation Parsing & Official Train/Validation/Test Split

### 05.1 Parse XML annotations and audit the data
TN5000 uses Pascal-VOC-style XML annotations. We keep only valid **benign/malignant** samples and verify the bounding boxes.

In [ ]:
TARGET_DATA_DIR = RAW / "TN5000_forReview"
images = list(TARGET_DATA_DIR.rglob("*.jpg")) + list(TARGET_DATA_DIR.rglob("*.jpeg")) + list(TARGET_DATA_DIR.rglob("*.JPG"))
xmls = list(TARGET_DATA_DIR.rglob("*.xml"))

lookup = {}
for p in images:
    lookup[p.name.lower()] = p
    lookup[p.stem.lower()] = p
print("images:", len(images), "xml:", len(xmls))

# TN5000 documentation uses BN/MN for benign/malignant; accept those plus common textual/numeric aliases.
LABEL_MAP = {
    "bn": "benign", "benign": "benign", "0": "benign",
    "mn": "malignant", "malignant": "malignant", "1": "malignant"
}

records = []
for xp in xmls:
    root = ET.parse(xp).getroot()
    fn = (root.findtext("filename") or "").strip()
    objs = root.findall("object")
    if not objs: continue
    o = objs[0]
    raw_label = (o.findtext("name") or "").strip().lower()
    label = LABEL_MAP.get(raw_label)
    bb = o.find("bndbox")
    if label is None or bb is None: continue
    try:
        box = [int(float(bb.findtext(k))) for k in ["xmin","ymin","xmax","ymax"]]
    except: continue
    ip = lookup.get(fn.lower()) or lookup.get(Path(fn).stem.lower()) or lookup.get(xp.stem.lower())
    if ip:
        records.append({"image_path": str(ip), "image_id": Path(ip).stem, "label": label,
                        "xmin": box[0], "ymin": box[1], "xmax": box[2], "ymax": box[3]})

df = pd.DataFrame(records).drop_duplicates(subset=["image_id"]).reset_index(drop=True)
print("usable:", len(df))
print(df.label.value_counts())
assert set(df.label.unique()) == {"benign", "malignant"}, "Expected both benign and malignant classes."

In [ ]:
def validate_and_clip_row(r):
    try:
        with Image.open(r.image_path) as im:
            w, h = im.size
        # If boundaries are completely scrambled or inverted, skip
        if r.xmin >= r.xmax or r.ymin >= r.ymax or r.xmin < 0 or r.ymin < 0:
            return None
        # Clip coordinates that slightly exceed image size boundaries
        r_copy = r.copy()
        r_copy['xmax'] = min(w, r.xmax)
        r_copy['ymax'] = min(h, r.ymax)
        return r_copy
    except:
        return None

processed_records = []
for _, row in df.iterrows():
    res = validate_and_clip_row(row)
    if res is not None:
        processed_records.append(res)

df = pd.DataFrame(processed_records).reset_index(drop=True)
print("after validation/clipping:", len(df))
print(df.label.value_counts())

### 05.2 Official TN5000 split — no random fallback
The TN5000 paper states that the benchmark provides an image-level **7:1:2** train/validation/test split: **3,500 / 500 / 1,000 images**. We use those official memberships from `ImageSets/Main/{train,val,test}.txt`.

**Important:** this notebook deliberately does **not** silently replace the official test set with a new random split. If the manifests are missing or cannot be matched, the cell raises an error so the release can be inspected before training.

In [ ]:
def find_split_file(candidates):
    hits=[]
    for p in RAW.rglob("*"):
        if p.is_file() and p.suffix.lower() in {".txt",".csv",".json"}:
            name=p.name.lower()
            if any(c in name for c in candidates): hits.append(p)
    return hits

def read_manifest(path):
    if path.suffix.lower() == ".txt":
        vals=[x.strip() for x in path.read_text(errors="ignore").splitlines() if x.strip() and not x.strip().startswith("#")]
    elif path.suffix.lower() == ".csv":
        z=pd.read_csv(path)
        col=next((c for c in z.columns if c.lower() in {"filename","file","image","image_id","path","name"}), z.columns[0])
        vals=z[col].astype(str).str.strip().tolist()
    else:
        z=json.loads(path.read_text(errors="ignore"))
        if isinstance(z,dict):
            for k in ["images","files","filenames","image_ids"]:
                if k in z: z=z[k]; break
        vals=[str(x).strip() for x in z if str(x).strip()]
    return vals

TARGET_DIR = RAW / "TN5000_forReview" / "ImageSets" / "Main"

manifest_candidates={
    "train":[p for p in TARGET_DIR.rglob("*.txt") if p.name.lower() in {"train.txt","training.txt"}],
    "val":[p for p in TARGET_DIR.rglob("*.txt") if p.name.lower() in {"val.txt","valid.txt","validation.txt","dev.txt"}],
    "test":[p for p in TARGET_DIR.rglob("*.txt") if p.name.lower() in {"test.txt","testing.txt","eval.txt","evaluation.txt"}]
}
print({k:[str(x.relative_to(RAW)) for x in v] for k,v in manifest_candidates.items()})

def manifest_to_ids(path):
    vals=read_manifest(path)
    ids=set()
    for v in vals:
        v=Path(v).stem
        ids.add(v.lower())
    return ids

if not all(manifest_candidates.values()):
    raise FileNotFoundError(
        "Official TN5000 train/val/test split manifests were not found in the downloaded release. "
        "Training is intentionally stopped instead of creating a random replacement split."
    )

train_ids=manifest_to_ids(manifest_candidates["train"][0])
val_ids=manifest_to_ids(manifest_candidates["val"][0])
test_ids=manifest_to_ids(manifest_candidates["test"][0])

assert not (train_ids & val_ids), "Train/validation overlap detected."
assert not (train_ids & test_ids), "Train/test overlap detected."
assert not (val_ids & test_ids), "Validation/test overlap detected."

df_ids=set(df.image_id.str.lower())
for name,ids in [("train",train_ids),("val",val_ids),("test",test_ids)]:
    missing=ids-df_ids
    if missing:
        raise ValueError(f"{name} manifest has {len(missing)} entries that could not be matched to parsed XML/image records. Example: {next(iter(missing))}")

train_df=df[df.image_id.str.lower().isin(train_ids)].copy().reset_index(drop=True)
val_df=df[df.image_id.str.lower().isin(val_ids)].copy().reset_index(drop=True)
test_df=df[df.image_id.str.lower().isin(test_ids)].copy().reset_index(drop=True)

print("Official split counts:")
for n,d in [("train",train_df),("val",val_df),("test",test_df)]:
    print(f"{n}: {len(d)}", d.label.value_counts().to_dict())

assert len(train_df)==3500 and len(val_df)==500 and len(test_df)==1000, \
    "TN5000 official split should contain exactly 3500/500/1000 images. Check manifest parsing."
assert len(set(train_df.image_id)&set(val_df.image_id))==0
assert len(set(train_df.image_id)&set(test_df.image_id))==0
assert len(set(val_df.image_id)&set(test_df.image_id))==0

print("Official TN5000 7:1:2 split verified successfully.")

# 06. ROI Preprocessing & Input Pipeline

### 06.1 ROI crops
The expert bounding box is used to remove irrelevant ultrasound background. A **15% context margin** is retained around the nodule. Original images are never overwritten. The exact same ROI pipeline is used for every model in this notebook.

In [ ]:
CROP=PROC/"roi_crops"; CROP.mkdir(parents=True,exist_ok=True); PAD=.15
def make_crops(d,split):
    out=CROP/split
    for c in ["benign","malignant"]: (out/c).mkdir(parents=True,exist_ok=True)
    paths=[]
    for i,r in d.iterrows():
        p=out/r.label/f"{i:05d}_{Path(r.image_path).stem}.jpg"
        if not p.exists():
            with Image.open(r.image_path).convert("RGB") as im:
                w,h=im.size; bw=r.xmax-r.xmin; bh=r.ymax-r.ymin
                px=int(bw*PAD); py=int(bh*PAD)
                box=(max(0,r.xmin-px),max(0,r.ymin-py),min(w,r.xmax+px),min(h,r.ymax+py))
                im.crop(box).save(p,quality=95)
        paths.append(str(p))
    return paths
train_df["crop_path"]=make_crops(train_df,"train")
val_df["crop_path"]=make_crops(val_df,"val")
test_df["crop_path"]=make_crops(test_df,"test")
print("ROI crops ready.")

In [ ]:
fig,ax=plt.subplots(2,4,figsize=(14,7))
for a,(_,r) in zip(ax.ravel(),train_df.sample(8,random_state=SEED).iterrows()):
    a.imshow(Image.open(r.crop_path)); a.set_title(r.label); a.axis("off")
plt.tight_layout(); plt.show()

### 06.2 Input pipeline, augmentation and class weights
Augmentation is **training-only**; validation and test data are deterministic (no shuffling, no augmentation). Because TN5000 has more malignant than benign images, **class weights** are used rather than discarding valid samples. Images are resized to 224×224 and passed through Keras VGG16 `preprocess_input` for all models.

In [ ]:
IMG=(224,224); BATCH=32; AUTO=tf.data.AUTOTUNE
aug=keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(.04),
    layers.RandomZoom(.10),
    layers.RandomTranslation(.05,.05),
    layers.RandomContrast(.10)
])
def ds_from(d,training=False):
    paths=d.crop_path.values
    y=d.label.map({"benign":0,"malignant":1}).values.astype("float32")
    ds=tf.data.Dataset.from_tensor_slices((paths,y))
    if training: ds=ds.shuffle(len(d),seed=SEED,reshuffle_each_iteration=True)
    def load(p,y):
        x=tf.image.decode_jpeg(tf.io.read_file(p),channels=3)
        x=tf.image.resize(x,IMG); x=tf.cast(x,tf.float32)
        if training: x=aug(x,training=True)
        return preprocess_input(x),y
    return ds.map(load,num_parallel_calls=AUTO).batch(BATCH).prefetch(AUTO)
train_ds=ds_from(train_df,True); val_ds=ds_from(val_df); test_ds=ds_from(test_df)
w=compute_class_weight("balanced",classes=np.array([0,1]),y=train_df.label.map({"benign":0,"malignant":1}))
class_weight={0:float(w[0]),1:float(w[1])}; print(class_weight)

# 07. VGG16T Baseline
Follows the paper's basic design: VGG-16 + BatchNorm + Dropout with a binary (single sigmoid unit) output. The ImageNet backbone is initially frozen. This is a **single model**, not an ensemble.

### 07.1 Model architecture

In [ ]:
def build_vgg16t():
    base=VGG16(weights="imagenet",include_top=False,input_shape=(224,224,3))
    base.trainable=False
    x=layers.Flatten()(base.output)
    x=layers.Dense(4096)(x); x=layers.BatchNormalization()(x); x=layers.Activation("relu")(x); x=layers.Dropout(.5)(x)
    x=layers.Dense(4096)(x); x=layers.BatchNormalization()(x); x=layers.Activation("relu")(x); x=layers.Dropout(.5)(x)
    out=layers.Dense(1,activation="sigmoid")(x)
    return keras.Model(base.input,out,name="VGG16T_single")
model=build_vgg16t()
model.compile(optimizer=keras.optimizers.Adam(1e-4),loss="binary_crossentropy",metrics=[keras.metrics.BinaryAccuracy(name="accuracy"),keras.metrics.AUC(name="auc")])
model.summary()

### 07.2 Training
Adam (1e-4), balanced class weights, up to 30 epochs. `ModelCheckpoint` keeps the **best checkpoint by validation AUC**; early stopping (patience 7) restores the best weights; `ReduceLROnPlateau` lowers the learning rate on plateau.

In [ ]:
CKPT = str(CHECKPOINT_DIR / "vgg16t_tn5000_baseline.keras")
cb=[
 keras.callbacks.ModelCheckpoint(CKPT,monitor="val_auc",mode="max",save_best_only=True,verbose=1),
 keras.callbacks.EarlyStopping(monitor="val_auc",mode="max",patience=7,restore_best_weights=True,verbose=1),
 keras.callbacks.ReduceLROnPlateau(monitor="val_auc",mode="max",factor=.25,patience=3,min_lr=1e-7,verbose=1)
]
hist=model.fit(train_ds,validation_data=val_ds,epochs=30,class_weight=class_weight,callbacks=cb)

### 07.3 Best checkpoint & test evaluation
The best checkpoint is reloaded from disk and evaluated once on the official test set (threshold 0.50).

In [ ]:
model=keras.models.load_model(CKPT)
yt=test_df.label.map({"benign":0,"malignant":1}).values
yp=model.predict(test_ds,verbose=1).ravel(); pred=(yp>=.5).astype(int)
cm=confusion_matrix(yt,pred); tn,fp,fn,tp=cm.ravel()
metrics={
"accuracy":accuracy_score(yt,pred),
"precision":precision_score(yt,pred,zero_division=0),
"recall":recall_score(yt,pred,zero_division=0),
"f1":f1_score(yt,pred,zero_division=0),
"auc":roc_auc_score(yt,yp),
"sensitivity":tp/(tp+fn),
"specificity":tn/(tn+fp)}
print(pd.Series(metrics).map(lambda x:f"{x:.4f}"))
print(classification_report(yt,pred,target_names=["Benign","Malignant"],digits=4))
sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",xticklabels=["Benign","Malignant"],yticklabels=["Benign","Malignant"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.show()

# 08. VGG16T Fine-Tuning
Only after recording the baseline, **VGG-16 block 5** is unfrozen and trained with a very small learning rate (Adam 1e-5). Earlier blocks and BatchNorm layers remain frozen. This tests whether domain-specific fine-tuning improves the single model without changing the dataset split.

### 08.1 Unfreeze block 5

In [ ]:
ft=keras.models.load_model(CKPT)

# Enable fine-tuning for block 5 layers directly on the flat model's layer hierarchy
trainable=False
for l in ft.layers:
    if l.name.startswith("block5_"):
        trainable=True
    l.trainable=trainable
    # Keep batch normalization layers completely frozen
    if isinstance(l,layers.BatchNormalization):
        l.trainable=False

ft.compile(optimizer=keras.optimizers.Adam(1e-5),loss="binary_crossentropy",metrics=[keras.metrics.BinaryAccuracy(name="accuracy"),keras.metrics.AUC(name="auc")])

### 08.2 Fine-tuning
Best checkpoint by validation AUC (`ModelCheckpoint`), early stopping (patience 6), `ReduceLROnPlateau`, up to 20 epochs, same class weights.

In [ ]:
FTCKPT = str(CHECKPOINT_DIR / "vgg16t_tn5000_finetuned.keras")
fcb=[
 keras.callbacks.ModelCheckpoint(FTCKPT,monitor="val_auc",mode="max",save_best_only=True,verbose=1),
 keras.callbacks.EarlyStopping(monitor="val_auc",mode="max",patience=6,restore_best_weights=True,verbose=1),
 keras.callbacks.ReduceLROnPlateau(monitor="val_auc",mode="max",factor=.25,patience=2,min_lr=1e-8,verbose=1)
]
hist_ft=ft.fit(train_ds,validation_data=val_ds,epochs=20,class_weight=class_weight,callbacks=fcb)

### 08.3 Test evaluation
The fine-tuned checkpoint is evaluated on the official test set (threshold 0.50) and compared with the baseline.

In [ ]:
ft=keras.models.load_model(FTCKPT)
yp2=ft.predict(test_ds,verbose=1).ravel(); pred2=(yp2>=.5).astype(int)
def score(y,p,prob):
    cm=confusion_matrix(y,p); tn,fp,fn,tp=cm.ravel()
    return [accuracy_score(y,p),precision_score(y,p,zero_division=0),recall_score(y,p,zero_division=0),f1_score(y,p,zero_division=0),roc_auc_score(y,prob),tp/(tp+fn),tn/(tn+fp)]
res=pd.DataFrame([score(yt,pred,yp),score(yt,pred2,yp2)],index=["Baseline VGG16T","Fine-tuned VGG16T"],columns=["Accuracy","Precision","Recall","F1","AUC","Sensitivity","Specificity"])
display((res*100).round(2))
print("\nAccuracy change:",round((res.loc["Fine-tuned VGG16T","Accuracy"]-res.loc["Baseline VGG16T","Accuracy"])*100,2),"percentage points")

In [ ]:
print("Baseline VGG16T confusion matrix:")
print(confusion_matrix(yt, pred))

print("\nFine-tuned VGG16T confusion matrix:")
print(confusion_matrix(yt, pred2))

In [ ]:
res_pct = (res * 100).round(2)
res_pct.to_csv(str(TABLE_DIR / "vgg16t_comparison.csv"))

# 09. ResNet50
Same dataset, ROI crops, augmentation, class weighting, callbacks logic and evaluation metrics as VGG16T. **Only the CNN architecture changes.**

### 09.1 Architecture

In [ ]:
# ============================================================
# RESNET50 — BASELINE MODEL
# Same dataset, ROI pipeline, augmentation and class weighting
# Only the CNN architecture changes
# ============================================================

def build_resnet50():
    base = ResNet50(
        weights="imagenet",
        include_top=False,
        input_shape=(IMG[0], IMG[1], 3)
    )

    # Freeze ImageNet backbone for baseline experiment
    base.trainable = False

    x = base.output
    x = layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
    x = layers.Dense(256, activation="relu", name="dense_256")(x)
    x = layers.Dropout(0.4, name="dropout")(x)

    output = layers.Dense(
        1,
        activation="sigmoid",
        name="classifier"
    )(x)

    model = keras.Model(
        inputs=base.input,
        outputs=output,
        name="ResNet50_single"
    )

    return model


resnet = build_resnet50()

resnet.summary()

### 09.2 Compile

In [ ]:
# ============================================================
# COMPILE RESNET50
# ============================================================

resnet.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.AUC(name="auc")
    ]
)

print("ResNet50 compiled successfully.")

### 09.3 Training
Adam (1e-4), up to 20 epochs, best checkpoint by validation AUC, early stopping (patience 6), `ReduceLROnPlateau`.

In [ ]:
# ============================================================
# TRAIN RESNET50 BASELINE
# ============================================================

RES_CKPT = str(CHECKPOINT_DIR / "resnet50_tn5000_best.keras")

res_callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=RES_CKPT,
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_auc",
        mode="max",
        factor=0.25,
        patience=2,
        min_lr=1e-8,
        verbose=1
    )
]

hist_res = resnet.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    class_weight=class_weight,
    callbacks=res_callbacks
)

### 09.4 Test evaluation

In [ ]:
# ============================================================
# RESNET50 BASELINE — FINAL TEST EVALUATION
# ============================================================

resnet = keras.models.load_model(RES_CKPT)

yp_res = resnet.predict(test_ds, verbose=1).ravel()
pred_res = (yp_res >= 0.5).astype(int)

resnet_scores = score(
    yt,
    pred_res,
    yp_res
)

resnet_result = pd.DataFrame(
    [resnet_scores],
    index=["ResNet50"],
    columns=[
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "AUC",
        "Sensitivity",
        "Specificity"
    ]
)

display((resnet_result * 100).round(2))

In [ ]:
# ============================================================
# CONFUSION MATRIX
# ============================================================

cm_res = confusion_matrix(yt, pred_res)

print("ResNet50 confusion matrix:")
print(cm_res)

tn, fp, fn, tp = cm_res.ravel()

print("\nTN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

# 10. Model Comparison
The test set is evaluated only after the baseline / fine-tuning / ResNet50 choices are frozen.

### 10.1 Load checkpoints and generate probabilities
All three best checkpoints are reloaded and used to generate validation and test probabilities, which feed every analysis from here on (comparison, confusion matrices, ROC, thresholds, bootstrap CIs, Grad-CAM). Inference is run under `tf.device('/CPU:0')` as in the original run, to avoid GPU out-of-memory errors when several large models are loaded.

In [ ]:
# ============================================================
# LOAD VGG16T CHECKPOINTS + VALIDATION PREDICTIONS
# Threshold selection will use ONLY the validation set.
# ============================================================

# Clear previous models and graphs from memory to free GPU space
keras.backend.clear_session()
gc.collect()

# Force CPU execution to guarantee we bypass any GPU Out-Of-Memory limitations
with tf.device('/CPU:0'):
    # Load the two trained models
    vgg_base = keras.models.load_model(CKPT, compile=False)
    vgg_ft = keras.models.load_model(FTCKPT, compile=False)

    # IMPORTANT:
    # val_ds must NOT be shuffled.
    # training=False in ds_from() ensures this.
    val_ds_eval = ds_from(val_df, training=False)

    # Ground-truth validation labels mapped to binary integers
    y_val = val_df["label"].map({"benign": 0, "malignant": 1}).values.astype(int)

    # Predictions
    p_val_base = vgg_base.predict(val_ds_eval, verbose=1).ravel()
    p_val_ft = vgg_ft.predict(val_ds_eval, verbose=1).ravel()

print("Validation samples:", len(y_val))
print("Baseline predictions:", len(p_val_base))
print("Fine-tuned predictions:", len(p_val_ft))

assert len(y_val) == len(p_val_base) == len(p_val_ft)

print("\nValidation predictions generated successfully on CPU.")

In [ ]:
# ============================================================
# TEST-SET PREDICTIONS — BASELINE & FINE-TUNED VGG16T
# ============================================================

# Create test dataset WITHOUT shuffling
test_ds_eval = ds_from(test_df, training=False)

# Ground-truth labels
y_test = test_df["label"].map({
    "benign": 0,
    "malignant": 1
}).values.astype(int)

print("Test samples:", len(y_test))

# ------------------------------------------------------------
# Generate test probabilities on CPU to match loaded models
# ------------------------------------------------------------

with tf.device('/CPU:0'):
    p_test_base = vgg_base.predict(
        test_ds_eval,
        verbose=1
    ).ravel()

    p_test_ft = vgg_ft.predict(
        test_ds_eval,
        verbose=1
    ).ravel()

print("Baseline test predictions:", len(p_test_base))
print("Fine-tuned test predictions:", len(p_test_ft))

assert len(y_test) == len(p_test_base) == len(p_test_ft)

In [ ]:
# ============================================================
# RESNET50 TEST PREDICTIONS
# ============================================================

# Free memory before loading ResNet50
keras.backend.clear_session()
gc.collect()

with tf.device("/CPU:0"):

    resnet50 = keras.models.load_model(
        RES_CKPT,
        compile=False
    )

    p_test_resnet = resnet50.predict(
        test_ds_eval,
        verbose=1
    ).ravel()

print("ResNet50 test predictions:", len(p_test_resnet))

assert len(y_test) == len(p_test_resnet)

print("ResNet50 test predictions generated successfully.")

### 10.2 Test-set metrics — all models (threshold 0.50)

In [ ]:
# ============================================================
# COMPLETE COMPARISON
# ============================================================

all_results = pd.concat([
    res,
    resnet_result
])

display((all_results * 100).round(2))

### 10.3 Confusion matrices — all models (threshold 0.50)

In [ ]:
predictions = {
    "Baseline VGG16T": (p_test_base >= 0.50).astype(int),
    "Fine-tuned VGG16T": (p_test_ft >= 0.50).astype(int),
    "ResNet50": (p_test_resnet >= 0.50).astype(int)
}

for name, preds in predictions.items():

    cm = confusion_matrix(y_test, preds)

    fig, ax = plt.subplots(figsize=(6, 5))

    im = ax.imshow(cm)

    ax.set_title(f"Confusion Matrix — {name}")
    ax.set_xlabel("Predicted Class")
    ax.set_ylabel("True Class")

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])

    ax.set_xticklabels(["Benign", "Malignant"])
    ax.set_yticklabels(["Benign", "Malignant"])

    for i in range(2):
        for j in range(2):
            ax.text(
                j,
                i,
                str(cm[i, j]),
                ha="center",
                va="center",
                fontsize=14
            )

    plt.colorbar(im, ax=ax)

    plt.tight_layout()

    safe_name = name.replace(" ", "_").replace("-", "")
    plt.savefig(
        os.path.join(FIG_DIR, f"Figure_Confusion_Matrix_{safe_name}.png"),
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

# 11. ROC Curves
ROC curves and AUC for all three models on the official test set.

In [ ]:
from sklearn.metrics import roc_curve, auc

roc_data = {
    "Baseline VGG16T": p_test_base,
    "Fine-tuned VGG16T": p_test_ft,
    "ResNet50": p_test_resnet
}

fig, ax = plt.subplots(figsize=(8, 7))

for name, probs in roc_data.items():
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)

    ax.plot(
        fpr,
        tpr,
        linewidth=2,
        label=f"{name} (AUC = {roc_auc:.4f})"
    )

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1
)

ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves for Thyroid Nodule Classification")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    os.path.join(FIG_DIR, "Figure_ROC_Curves.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# 12. Threshold Analysis
Alternative operating points are explored **without touching the test set for selection**:
1. A grid of 161 thresholds in [0.10, 0.90] is scanned on the **validation** probabilities.
2. The threshold maximising **balanced accuracy** is selected per model.
3. That threshold is **locked** and applied once to the test probabilities; results are compared with the default 0.50.

In [ ]:
# ============================================================
# SELECT THRESHOLD USING VALIDATION SET ONLY
# ============================================================

def find_best_threshold(y_true, probabilities):
    thresholds = np.linspace(0.10, 0.90, 161)
    rows = []

    for threshold in thresholds:
        predictions = (probabilities >= threshold).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_true, predictions
        ).ravel()

        sensitivity = tp / (tp + fn)
        specificity = tn / (tn + fp)

        rows.append({
            "threshold": threshold,
            "accuracy": accuracy_score(y_true, predictions),
            "balanced_accuracy": balanced_accuracy_score(
                y_true, predictions
            ),
            "precision": precision_score(
                y_true, predictions, zero_division=0
            ),
            "sensitivity": sensitivity,
            "specificity": specificity,
            "f1": f1_score(
                y_true, predictions, zero_division=0
            )
        })

    results = pd.DataFrame(rows)

    # IMPORTANT:
    # Threshold is selected using VALIDATION SET ONLY.
    # We maximize balanced accuracy rather than test accuracy.
    best_idx = results["balanced_accuracy"].idxmax()

    return (
        float(results.loc[best_idx, "threshold"]),
        results
    )


# Find validation-optimal thresholds
best_threshold_base, threshold_base_df = find_best_threshold(
    y_val,
    p_val_base
)

best_threshold_ft, threshold_ft_df = find_best_threshold(
    y_val,
    p_val_ft
)


print("Validation-selected thresholds")
print("--------------------------------")
print(f"Baseline VGG16T   : {best_threshold_base:.3f}")
print(f"Fine-tuned VGG16T : {best_threshold_ft:.3f}")

In [ ]:
# ============================================================
# VALIDATION PERFORMANCE AT SELECTED THRESHOLDS
# ============================================================

def threshold_row(df, threshold):
    idx = np.abs(df["threshold"] - threshold).argmin()
    return df.iloc[idx]


val_base_best = threshold_row(
    threshold_base_df,
    best_threshold_base
)

val_ft_best = threshold_row(
    threshold_ft_df,
    best_threshold_ft
)


val_threshold_comparison = pd.DataFrame(
    [val_base_best, val_ft_best],
    index=[
        "Baseline VGG16T",
        "Fine-tuned VGG16T"
    ]
)

display(val_threshold_comparison.round(4))

In [ ]:
# ============================================================
# APPLY LOCKED VALIDATION THRESHOLDS TO TEST SET
# ============================================================

# IMPORTANT:
# The thresholds 0.580 and 0.585 were selected ONLY using
# the validation set. They are now locked.

# ------------------------------------------------------------
# Apply LOCKED thresholds
# ------------------------------------------------------------

pred_base_tuned = (
    p_test_base >= best_threshold_base
).astype(int)

pred_ft_tuned = (
    p_test_ft >= best_threshold_ft
).astype(int)

print("\nLocked thresholds applied:")
print(f"Baseline VGG16T   : {best_threshold_base:.3f}")
print(f"Fine-tuned VGG16T : {best_threshold_ft:.3f}")

print("\nTest-set evaluation is now ready.")

In [ ]:
# ============================================================
# COMPARE TEST PERFORMANCE
# ============================================================

def evaluate_predictions(y_true, predictions, probabilities):

    cm = confusion_matrix(y_true, predictions)

    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)

    return {
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_true, predictions, zero_division=0
        ),
        "AUC": roc_auc_score(
            y_true, probabilities
        ),
        "Sensitivity": sensitivity,
        "Specificity": specificity
    }


# ------------------------------------------------------------
# Original fixed threshold = 0.50
# ------------------------------------------------------------

base_050 = evaluate_predictions(
    y_test,
    (p_test_base >= 0.50).astype(int),
    p_test_base
)

ft_050 = evaluate_predictions(
    y_test,
    (p_test_ft >= 0.50).astype(int),
    p_test_ft
)


# ------------------------------------------------------------
# Validation-selected thresholds
# ------------------------------------------------------------

base_val_threshold = evaluate_predictions(
    y_test,
    pred_base_tuned,
    p_test_base
)

ft_val_threshold = evaluate_predictions(
    y_test,
    pred_ft_tuned,
    p_test_ft
)


# ------------------------------------------------------------
# Final comparison
# ------------------------------------------------------------

threshold_comparison = pd.DataFrame(
    [
        base_050,
        base_val_threshold,
        ft_050,
        ft_val_threshold
    ],
    index=[
        "Baseline VGG16T @ 0.500",
        f"Baseline VGG16T @ {best_threshold_base:.3f}",
        "Fine-tuned VGG16T @ 0.500",
        f"Fine-tuned VGG16T @ {best_threshold_ft:.3f}"
    ]
)

display(
    (threshold_comparison * 100).round(2)
)

# 13. Bootstrap 95% Confidence Intervals
Percentile 95% CIs from 2,000 bootstrap resamples of the 1,000 test images (threshold 0.50; seeds 42 / 43 / 44 for baseline / fine-tuned / ResNet50). Resamples missing a class are skipped (none were in this run: 2,000/2,000 valid).

In [ ]:
# ============================================================
# 95% BOOTSTRAP CONFIDENCE INTERVALS
# ============================================================

# ------------------------------------------------------------
# Bootstrap configuration
# ------------------------------------------------------------

N_BOOTSTRAPS = 2000
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)


# ------------------------------------------------------------
# Metric calculation
# ------------------------------------------------------------

def calculate_metrics(y_true, probabilities, threshold=0.50):

    predictions = (
        probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    return {
        "Accuracy": accuracy_score(
            y_true,
            predictions
        ),

        "Precision": precision_score(
            y_true,
            predictions,
            zero_division=0
        ),

        "Sensitivity": sensitivity,

        "Specificity": specificity,

        "F1": f1_score(
            y_true,
            predictions,
            zero_division=0
        ),

        "AUC": roc_auc_score(
            y_true,
            probabilities
        )
    }


# ------------------------------------------------------------
# Bootstrap CI
# ------------------------------------------------------------

def bootstrap_ci(
    y_true,
    probabilities,
    threshold=0.50,
    n_bootstraps=2000,
    seed=42
):

    rng = np.random.default_rng(seed)

    n = len(y_true)

    metric_names = [
        "Accuracy",
        "Precision",
        "Sensitivity",
        "Specificity",
        "F1",
        "AUC"
    ]

    bootstrap_results = {
        metric: []
        for metric in metric_names
    }

    valid_iterations = 0

    for _ in range(n_bootstraps):

        # Sample test observations with replacement
        indices = rng.integers(
            0,
            n,
            size=n
        )

        y_boot = y_true[indices]
        p_boot = probabilities[indices]

        # AUC requires both classes to be present
        if len(np.unique(y_boot)) < 2:
            continue

        metrics = calculate_metrics(
            y_boot,
            p_boot,
            threshold
        )

        for metric in metric_names:
            bootstrap_results[metric].append(
                metrics[metric]
            )

        valid_iterations += 1

    # --------------------------------------------------------
    # Percentile 95% CI
    # --------------------------------------------------------

    results = {}

    original_metrics = calculate_metrics(
        y_true,
        probabilities,
        threshold
    )

    for metric in metric_names:

        values = np.asarray(
            bootstrap_results[metric]
        )

        lower = np.percentile(
            values,
            2.5
        )

        upper = np.percentile(
            values,
            97.5
        )

        results[metric] = {
            "value": original_metrics[metric],
            "lower": lower,
            "upper": upper
        }

    print(
        f"Valid bootstrap samples: "
        f"{valid_iterations}/{n_bootstraps}"
    )

    return results


# ============================================================
# Calculate CIs for all three models
# ============================================================

ci_base = bootstrap_ci(
    y_test,
    p_test_base,
    threshold=0.50,
    n_bootstraps=N_BOOTSTRAPS,
    seed=42
)

ci_ft = bootstrap_ci(
    y_test,
    p_test_ft,
    threshold=0.50,
    n_bootstraps=N_BOOTSTRAPS,
    seed=43
)

ci_resnet = bootstrap_ci(
    y_test,
    p_test_resnet,
    threshold=0.50,
    n_bootstraps=N_BOOTSTRAPS,
    seed=44
)

print("\nBootstrap confidence intervals calculated successfully.")

In [ ]:
# ============================================================
# PUBLICATION-READY 95% CI TABLE
# ============================================================

def format_ci(ci_dict):

    rows = {}

    for metric, values in ci_dict.items():

        rows[metric] = (
            f"{values['value'] * 100:.2f}% "
            f"({values['lower'] * 100:.2f}–"
            f"{values['upper'] * 100:.2f}%)"
        )

    return rows


ci_table = pd.DataFrame({
    "Baseline VGG16T": format_ci(ci_base),
    "Fine-tuned VGG16T": format_ci(ci_ft),
    "ResNet50": format_ci(ci_resnet)
})

display(ci_table)

# 14. Grad-CAM Explainability
Grad-CAM on the **baseline VGG16T** (target layer `block5_conv3`, 14×14 feature maps), computed on the ROI **crops** used for the research models and VGG16-preprocessed like the training pipeline.

The cell first re-evaluates the baseline on the official test set and prints the metrics as a consistency check against the reported result, then visualises six cases: the first four correctly classified test images, the first false positive and the first false negative.

Notes:
- Grad-CAM is **qualitative** only; it is not a validation of the model's clinical reasoning.
- The explanation images are loaded with OpenCV (`cv2.resize`), whereas training/evaluation use `tf.image` decoding/resizing. Probabilities printed in the figure can therefore differ slightly (in the third decimal) from the dataset-pipeline probabilities used in the evaluation tables; the class decision is unaffected for the six cases shown.

In [ ]:
# ============================================================
# GRAD-CAM — VGG16T BASELINE (CORRECTED IMPLEMENTATION)
# ============================================================

# ------------------------------------------------------------
# 1. Existing trained model
# ------------------------------------------------------------

MODEL_PATH = CKPT

model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)

IMG_SIZE = (224, 224)
GRAD_LAYER = "block5_conv3"

print("Model:", model.name)
print("Grad-CAM layer:", GRAD_LAYER)
print("Test samples:", len(test_df))

# ------------------------------------------------------------
# 2. IMPORTANT: use the SAME preprocessing as original
# ------------------------------------------------------------

assert "crop_path" in test_df.columns, (
    "crop_path is missing from test_df."
)

assert "label" in test_df.columns, (
    "label is missing from test_df."
)

# Use the exact existing ds_from()
test_ds = ds_from(test_df, training=False)

# Ground truth
y_test_exact = (
    test_df["label"]
    .map({"benign": 0, "malignant": 1})
    .values.astype(int)
)

# ------------------------------------------------------------
# 3. Predict the complete official test set
# ------------------------------------------------------------

print("\nRunning prediction on official test set...")

p_test_exact = model.predict(
    test_ds,
    verbose=1
).ravel()

y_pred_exact = (
    p_test_exact >= 0.50
).astype(int)

# ------------------------------------------------------------
# 4. Verify metrics against original reported result
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_test_exact,
    y_pred_exact
)

precision = precision_score(
    y_test_exact,
    y_pred_exact,
    zero_division=0
)

recall = recall_score(
    y_test_exact,
    y_pred_exact,
    zero_division=0
)

f1 = f1_score(
    y_test_exact,
    y_pred_exact,
    zero_division=0
)

auc = roc_auc_score(
    y_test_exact,
    p_test_exact
)

tn, fp, fn, tp = confusion_matrix(
    y_test_exact,
    y_pred_exact
).ravel()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

print("\n" + "=" * 60)
print("EXACT TEST-SET VERIFICATION")
print("=" * 60)

print(f"Accuracy       : {accuracy*100:.2f}%")
print(f"Precision      : {precision*100:.2f}%")
print(f"Recall         : {recall*100:.2f}%")
print(f"F1             : {f1*100:.2f}%")
print(f"AUC            : {auc*100:.2f}%")
print(f"Sensitivity    : {sensitivity*100:.2f}%")
print(f"Specificity    : {specificity*100:.2f}%")

print("\nConfusion matrix:")
print(confusion_matrix(
    y_test_exact,
    y_pred_exact
))

# ------------------------------------------------------------
# 5. Check against our reported baseline
# ------------------------------------------------------------

print("\nExpected original result:")
print("Accuracy    ≈ 87.40%")
print("Sensitivity ≈ 91.38%")
print("Specificity ≈ 76.58%")
print("AUC         ≈ 90.58%")

# ------------------------------------------------------------
# 6. Find exact FP / FN / correct cases
# ------------------------------------------------------------

correct_indices = np.where(
    y_test_exact == y_pred_exact
)[0]

false_positive_indices = np.where(
    (y_test_exact == 0) &
    (y_pred_exact == 1)
)[0]

false_negative_indices = np.where(
    (y_test_exact == 1) &
    (y_pred_exact == 0)
)[0]

print("\nExact classification counts:")
print("Correct       :", len(correct_indices))
print("False positive:", len(false_positive_indices))
print("False negative:", len(false_negative_indices))

# ------------------------------------------------------------
# 7. Select 4 correct + 1 FP + 1 FN
# ------------------------------------------------------------

selected_indices = [
    int(correct_indices[0]),
    int(correct_indices[1]),
    int(correct_indices[2]),
    int(correct_indices[3]),
    int(false_positive_indices[0]),
    int(false_negative_indices[0])
]

print("\nSelected Grad-CAM indices:")
print(selected_indices)

# ------------------------------------------------------------
# 8. Grad-CAM model
# ------------------------------------------------------------

grad_model = tf.keras.models.Model(
    inputs=model.inputs,
    outputs=[
        model.get_layer(GRAD_LAYER).output,
        model.output
    ]
)

# ------------------------------------------------------------
# 9. Exact crop-path image loader
# ------------------------------------------------------------

def load_crop_exact(path):

    img = cv2.imread(str(path))

    if img is None:
        raise FileNotFoundError(
            f"Could not read image: {path}"
        )

    img = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2RGB
    )

    # EXACT SAME operation as ds_from()
    resized = cv2.resize(
        img,
        IMG_SIZE
    )

    x = resized.astype(
        np.float32
    )

    # EXACT SAME preprocessing as ds_from()
    x = tf.keras.applications.vgg16.preprocess_input(x)

    x = np.expand_dims(
        x,
        axis=0
    )

    return resized, x


# ------------------------------------------------------------
# 10. Grad-CAM function
# ------------------------------------------------------------

def make_gradcam(img_tensor):

    with tf.GradientTape() as tape:

        conv_outputs, prediction = grad_model(
            img_tensor,
            training=False
        )

        score = prediction[:, 0]

    gradients = tape.gradient(
        score,
        conv_outputs
    )

    pooled_gradients = tf.reduce_mean(
        gradients,
        axis=(1, 2)
    )

    conv_outputs = conv_outputs[0]
    pooled_gradients = pooled_gradients[0]

    heatmap = tf.reduce_sum(
        conv_outputs * pooled_gradients,
        axis=-1
    )

    heatmap = tf.maximum(
        heatmap,
        0
    )

    max_value = tf.reduce_max(
        heatmap
    )

    heatmap = tf.where(
        max_value > 0,
        heatmap / max_value,
        heatmap
    )

    return heatmap.numpy()


# ------------------------------------------------------------
# 11. Generate final figure
# ------------------------------------------------------------

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15, 9)
)

axes = axes.flatten()

gradcam_results = []

for plot_idx, df_idx in enumerate(
    selected_indices
):

    row = test_df.iloc[df_idx]

    # IMPORTANT:
    # Use crop_path — exactly as original evaluation
    image_path = row["crop_path"]

    true_label = str(
        row["label"]
    ).lower()

    # Load exact cropped input
    display_img, img_tensor = load_crop_exact(
        image_path
    )

    # Prediction from the exact same input
    probability = float(
        model.predict(
            img_tensor,
            verbose=0
        )[0][0]
    )

    predicted_label = (
        "malignant"
        if probability >= 0.50
        else "benign"
    )

    # Grad-CAM
    heatmap = make_gradcam(
        img_tensor
    )

    heatmap = cv2.resize(
        heatmap,
        IMG_SIZE
    )

    heatmap_uint8 = np.uint8(
        255 * heatmap
    )

    heatmap_color = cv2.applyColorMap(
        heatmap_uint8,
        cv2.COLORMAP_JET
    )

    heatmap_color = cv2.cvtColor(
        heatmap_color,
        cv2.COLOR_BGR2RGB
    )

    overlay = cv2.addWeighted(
        display_img,
        0.60,
        heatmap_color,
        0.40,
        0
    )

    # Classification type
    if true_label == predicted_label:
        case_type = "Correct"
    elif (
        true_label == "benign"
        and predicted_label == "malignant"
    ):
        case_type = "False Positive"
    else:
        case_type = "False Negative"

    axes[plot_idx].imshow(
        overlay
    )

    axes[plot_idx].axis(
        "off"
    )

    axes[plot_idx].set_title(
        f"True: {true_label.capitalize()}\n"
        f"Pred: {predicted_label.capitalize()} "
        f"(P(Malignant)={probability:.4f})\n"
        f"{case_type}",
        fontsize=11
    )

    gradcam_results.append({
        "test_index": df_idx,
        "true": true_label,
        "prediction": predicted_label,
        "p_malignant": probability,
        "type": case_type,
        "crop_path": image_path
    })

plt.suptitle(
    "Grad-CAM Visualization of VGG16T on TN5000 Test Images",
    fontsize=16,
    y=0.98
)

plt.tight_layout()

# ------------------------------------------------------------
# 12. Save final figure
# ------------------------------------------------------------

local_output = FIG_DIR / "Figure_GradCAM_Qualitative_Analysis.png"

plt.savefig(
    local_output,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 13. Display exact cases
# ------------------------------------------------------------

gradcam_df = pd.DataFrame(
    gradcam_results
)

print("\nFinal Grad-CAM cases:")
display(
    gradcam_df[
        [
            "test_index",
            "true",
            "prediction",
            "p_malignant",
            "type"
        ]
    ]
)

# ------------------------------------------------------------
# 14. Final note
# ------------------------------------------------------------
print("\nGrad-CAM figure saved to:", local_output)


# 15. Model Parameters & Efficiency

### 15.1 Parameter counts

In [ ]:
# ============================================================
# MODEL SIZE / PARAMETER COUNT
# ============================================================

def model_parameter_summary(model, name):

    total_params = model.count_params()

    trainable_params = np.sum([
        np.prod(v.shape)
        for v in model.trainable_variables
    ])

    non_trainable_params = (
        total_params - trainable_params
    )

    return {
        "Model": name,
        "Total Parameters": int(total_params),
        "Trainable Parameters": int(trainable_params),
        "Non-trainable Parameters": int(non_trainable_params)
    }


parameter_table = pd.DataFrame([
    model_parameter_summary(
        vgg_base,
        "Baseline VGG16T"
    ),

    model_parameter_summary(
        vgg_ft,
        "Fine-tuned VGG16T"
    ),

    model_parameter_summary(
        resnet50,
        "ResNet50"
    )
])

display(parameter_table)

### 15.2 Checkpoint file sizes

In [ ]:
# ============================================================
# CHECK CHECKPOINT FILE SIZES
# ============================================================

checkpoint_paths = {
    "Baseline VGG16T": CKPT,
    "Fine-tuned VGG16T": FTCKPT,
    "ResNet50": RES_CKPT
}

size_rows = []

for model_name, path in checkpoint_paths.items():

    if os.path.exists(path):

        size_mb = (
            os.path.getsize(path)
            / (1024 ** 2)
        )

        size_rows.append({
            "Model": model_name,
            "Checkpoint Size (MB)": size_mb,
            "Checkpoint Size (GB)": size_mb / 1024
        })

    else:

        print(
            f"WARNING: checkpoint not found:\n{path}"
        )


checkpoint_table = pd.DataFrame(
    size_rows
)

display(
    checkpoint_table.round(2)
)

### 15.3 CPU inference benchmark
Forward-pass speed on **CPU** for the three research models, measured on the first 128 images of the official test set (8-image warm-up, then one timed `predict` call over all 128 images). Absolute timings depend on the host machine.

In [ ]:
# ============================================================
# CPU INFERENCE BENCHMARK — 128 official-test images
# ============================================================

# 1. Benchmark images: first 128 images (4 batches) of the official test set
benchmark_images = []
for batch_images, _ in ds_from(test_df, training=False).take(4):
    benchmark_images.append(batch_images.numpy())
benchmark_images = np.concatenate(benchmark_images, axis=0)[:128]

print("Benchmark images ready:", benchmark_images.shape)

# 2. Load models on CPU
with tf.device("/CPU:0"):
    vgg_base = keras.models.load_model(CKPT, compile=False)
    vgg_ft = keras.models.load_model(FTCKPT, compile=False)
    resnet50 = keras.models.load_model(RES_CKPT, compile=False)

# 3. Benchmarking
def measure_quick(model, images, name):
    with tf.device("/CPU:0"):
        model.predict(images[:8], verbose=0)
    start = time.perf_counter()
    with tf.device("/CPU:0"):
        predictions = model.predict(images, verbose=0)
    elapsed = time.perf_counter() - start
    n = len(predictions)
    return {
        "Model": name,
        "Images": n,
        "Total Time (s)": elapsed,
        "Time/Image (ms)": (elapsed / n) * 1000,
        "Images/Second": n / elapsed,
    }

results = []
results.append(measure_quick(vgg_base, benchmark_images, "Baseline VGG16T"))
results.append(measure_quick(vgg_ft, benchmark_images, "Fine-tuned VGG16T"))
results.append(measure_quick(resnet50, benchmark_images, "ResNet50"))

inference_table = pd.DataFrame(results)
display(inference_table.round(2))

### 15.4 Size & speed tables and figures
Values below are transcribed from the measured outputs in 15.1–15.3 (parameter counts, checkpoint sizes, CPU timings) and saved as CSV / PNG.

In [ ]:
model_size = pd.DataFrame({
    "Model": [
        "Baseline VGG16T",
        "Fine-tuned VGG16T",
        "ResNet50"
    ],
    "Parameters_Million": [
        134.297409,
        134.297409,
        24.112513
    ],
    "Checkpoint_MB": [
        1424.63,
        1478.52,
        96.61
    ]
})

model_size.to_csv(
    os.path.join(TABLE_DIR, "Table_Model_Size.csv"),
    index=False
)

fig, ax = plt.subplots(figsize=(9, 6))

x = np.arange(len(model_size))

ax.bar(
    x - 0.2,
    model_size["Parameters_Million"],
    width=0.4,
    label="Parameters (million)"
)

ax.set_xticks(x)
ax.set_xticklabels(model_size["Model"])
ax.set_ylabel("Parameters (million)")
ax.set_title("Model Parameter Count")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()

plt.savefig(
    os.path.join(FIG_DIR, "Figure_Model_Parameter_Count.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

ax.bar(
    model_size["Model"],
    model_size["Checkpoint_MB"]
)

ax.set_ylabel("Checkpoint Size (MB)")
ax.set_title("Model Checkpoint Size")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()

plt.savefig(
    os.path.join(FIG_DIR, "Figure_Model_Checkpoint_Size.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
inference_results = pd.DataFrame({
    "Model": [
        "Baseline VGG16T",
        "Fine-tuned VGG16T",
        "ResNet50"
    ],
    "Time_per_image_ms": [
        346.88,
        640.94,
        129.26
    ],
    "Images_per_second": [
        2.88,
        1.56,
        7.74
    ]
})

inference_results.to_csv(
    os.path.join(TABLE_DIR, "Table_CPU_Inference_Benchmark.csv"),
    index=False
)

fig, ax = plt.subplots(figsize=(9, 6))

ax.bar(
    inference_results["Model"],
    inference_results["Images_per_second"]
)

ax.set_ylabel("Images per second")
ax.set_title("CPU Inference Throughput")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()

plt.savefig(
    os.path.join(FIG_DIR, "Figure_CPU_Inference_Throughput.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# 16. Final Results Tables & Figures

Publication tables and figures generated from the computed evaluation outputs. Files are saved under the local `results/tables/` and `results/figures/` directories.

### 16.1 Model performance (threshold 0.50)


In [ ]:
model_results = pd.DataFrame([
    {
        "Model": "Baseline VGG16T",
        "Accuracy": 0.8740,
        "Precision": 0.9138,
        "Sensitivity": 0.9138,
        "Specificity": 0.7658,
        "F1": 0.9138,
        "AUC": 0.9058
    },
    {
        "Model": "Fine-tuned VGG16T",
        "Accuracy": 0.8450,
        "Precision": 0.9350,
        "Sensitivity": 0.8468,
        "Specificity": 0.8401,
        "F1": 0.8887,
        "AUC": 0.9216
    },
    {
        "Model": "ResNet50",
        "Accuracy": 0.8410,
        "Precision": 0.9231,
        "Sensitivity": 0.8536,
        "Specificity": 0.8067,
        "F1": 0.8870,
        "AUC": 0.9135
    }
])

model_results_percent = model_results.copy()

for col in model_results.columns[1:]:
    model_results_percent[col] *= 100

model_results_percent.to_csv(
    os.path.join(TABLE_DIR, "Table_Model_Performance.csv"),
    index=False
)

display(model_results_percent.round(2))

In [ ]:
metrics = [
    "Accuracy",
    "Precision",
    "Sensitivity",
    "Specificity",
    "F1",
    "AUC"
]

x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 7))

for i, row in model_results_percent.iterrows():
    values = [row[m] for m in metrics]
    ax.bar(
        x + (i - 1) * width,
        values,
        width,
        label=row["Model"]
    )

ax.set_ylabel("Percentage (%)")
ax.set_title("Performance Comparison of Deep Learning Models")
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 100)
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()

plt.savefig(
    os.path.join(FIG_DIR, "Figure_Model_Performance_Comparison.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### 16.2 Bootstrap 95% confidence intervals

In [ ]:
ci_results = pd.DataFrame([
    ["Baseline VGG16T", "87.40", "85.40–89.40", "91.38", "89.33–93.44",
     "91.38", "89.31–93.36", "76.58", "71.15–81.64",
     "91.38", "89.89–92.86", "90.58", "88.21–92.84"],

    ["Fine-tuned VGG16T", "84.50", "82.30–86.70", "93.50", "91.62–95.31",
     "84.68", "81.99–87.21", "84.01", "79.68–88.38",
     "88.87", "87.08–90.55", "92.16", "90.24–94.05"],

    ["ResNet50", "84.10", "81.70–86.30", "92.31", "90.25–94.31",
     "85.36", "82.72–87.87", "80.67", "75.85–85.26",
     "88.70", "86.82–90.41", "91.35", "89.24–93.25"]
], columns=[
    "Model",
    "Accuracy (%)", "Accuracy 95% CI",
    "Precision (%)", "Precision 95% CI",
    "Sensitivity (%)", "Sensitivity 95% CI",
    "Specificity (%)", "Specificity 95% CI",
    "F1 (%)", "F1 95% CI",
    "AUC (%)", "AUC 95% CI"
])

ci_results.to_csv(
    os.path.join(TABLE_DIR, "Table_95_Bootstrap_CI.csv"),
    index=False
)

display(ci_results)

In [ ]:
ci_plot = pd.DataFrame([
    ["Baseline VGG16T", "Accuracy", 87.40, 85.40, 89.40],
    ["Fine-tuned VGG16T", "Accuracy", 84.50, 82.30, 86.70],
    ["ResNet50", "Accuracy", 84.10, 81.70, 86.30],

    ["Baseline VGG16T", "Sensitivity", 91.38, 89.31, 93.36],
    ["Fine-tuned VGG16T", "Sensitivity", 84.68, 81.99, 87.21],
    ["ResNet50", "Sensitivity", 85.36, 82.72, 87.87],

    ["Baseline VGG16T", "Specificity", 76.58, 71.15, 81.64],
    ["Fine-tuned VGG16T", "Specificity", 84.01, 79.68, 88.38],
    ["ResNet50", "Specificity", 80.67, 75.85, 85.26],

    ["Baseline VGG16T", "AUC", 90.58, 88.21, 92.84],
    ["Fine-tuned VGG16T", "AUC", 92.16, 90.24, 94.05],
    ["ResNet50", "AUC", 91.35, 89.24, 93.25]
], columns=["Model", "Metric", "Value", "Lower", "Upper"])

fig, ax = plt.subplots(figsize=(11, 7))

metrics_ci = ["Accuracy", "Sensitivity", "Specificity", "AUC"]

x_positions = np.arange(len(metrics_ci))
offsets = [-0.25, 0, 0.25]

for model_idx, model in enumerate(
    ["Baseline VGG16T", "Fine-tuned VGG16T", "ResNet50"]
):
    subset = ci_plot[ci_plot["Model"] == model]

    vals = [
        subset[subset["Metric"] == metric]["Value"].iloc[0]
        for metric in metrics_ci
    ]

    lowers = [
        subset[subset["Metric"] == metric]["Lower"].iloc[0]
        for metric in metrics_ci
    ]

    uppers = [
        subset[subset["Metric"] == metric]["Upper"].iloc[0]
        for metric in metrics_ci
    ]

    yerr = [
        np.array(vals) - np.array(lowers),
        np.array(uppers) - np.array(vals)
    ]

    ax.errorbar(
        x_positions + offsets[model_idx],
        vals,
        yerr=yerr,
        fmt="o",
        capsize=5,
        linewidth=1.5,
        label=model
    )

ax.set_xticks(x_positions)
ax.set_xticklabels(metrics_ci)
ax.set_ylabel("Percentage (%)")
ax.set_title("Performance Estimates with 95% Bootstrap Confidence Intervals")
ax.set_ylim(60, 100)
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()

plt.savefig(
    os.path.join(FIG_DIR, "Figure_95_Bootstrap_Confidence_Intervals.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### 16.3 Threshold analysis (0.50 vs validation-selected threshold)

In [ ]:
threshold_results = pd.DataFrame([
    ["Baseline VGG16T", 0.500, 87.40, 91.38, 91.38, 91.38, 76.58],
    ["Baseline VGG16T", 0.580, 87.20, 91.93, 90.42, 91.17, 78.44],
    ["Fine-tuned VGG16T", 0.500, 84.50, 93.50, 84.68, 88.87, 84.01],
    ["Fine-tuned VGG16T", 0.585, 83.50, 93.67, 83.04, 88.03, 84.76]
], columns=[
    "Model",
    "Threshold",
    "Accuracy",
    "Precision",
    "Sensitivity",
    "F1",
    "Specificity"
])

threshold_results.to_csv(
    os.path.join(TABLE_DIR, "Table_Threshold_Analysis.csv"),
    index=False
)

display(threshold_results)

### 16.4 Local research artifact archive

The notebook already writes figures and tables to `results/`. This optional cell creates a simple `results/release/` folder containing the trained checkpoints and key publication tables, making the outputs easy to package with a code release.


In [ ]:
# Optional: create a compact local release folder
RELEASE_DIR = OUTPUT_ROOT / "release"
RELEASE_DIR.mkdir(parents=True, exist_ok=True)

artifacts = [
    CKPT,
    FTCKPT,
    RES_CKPT,
    str(TABLE_DIR / "Table_Model_Performance.csv"),
    str(TABLE_DIR / "Table_95_Bootstrap_CI.csv"),
    str(TABLE_DIR / "Table_Threshold_Analysis.csv"),
]

for artifact in artifacts:
    artifact_path = Path(artifact)
    if artifact_path.exists():
        shutil.copy2(artifact_path, RELEASE_DIR / artifact_path.name)
        print("Copied:", artifact_path.name)
    else:
        print("Skipped (not found):", artifact_path)

print("\nRelease folder:", RELEASE_DIR)


# 17. Research Conclusions

All statements below restate numbers computed in this notebook (official TN5000 test set, n = 1,000; default threshold 0.50 unless stated).

**Headline results**

| Model | Accuracy | Sensitivity | Specificity | F1 | AUC (95% CI) |
|---|---|---|---|---|---|
| Baseline VGG16T | **87.40%** | **91.38%** | 76.58% | **91.38%** | 90.58% (88.21–92.84) |
| Fine-tuned VGG16T | 84.50% | 84.68% | **84.01%** | 88.87% | **92.16%** (90.24–94.05) |
| ResNet50 | 84.10% | 85.36% | 80.67% | 88.70% | 91.35% (89.24–93.25) |

**Findings**
1. **Baseline VGG16T had the best accuracy, sensitivity and F1** at the default threshold. Unfreezing block 5 did not improve accuracy (−2.90 pp). It raised AUC (+1.58 pp) and specificity (+7.43 pp) but reduced sensitivity (−6.70 pp), i.e. it shifted the operating balance rather than uniformly improving the model. Validation AUC peaked at the first fine-tuning epoch (0.9544), so the selected fine-tuned checkpoint is the epoch-1 weights.
2. **ResNet50 under the identical pipeline** reached 84.10% accuracy and 91.35% AUC — comparable discrimination to both VGG16T variants while using far fewer parameters.
3. **Statistical uncertainty is substantial.** The bootstrap 95% CIs of AUC overlap across all three models, and accuracy CIs overlap pairwise. No paired significance test (e.g. DeLong, McNemar) was run, so the AUC ranking should not be read as established.
4. **Validation-selected thresholds (0.580 baseline, 0.585 fine-tuned) gave no accuracy benefit** on the test set (87.20% and 83.50%). They traded a little sensitivity for a little specificity (baseline 78.44% specificity / 90.42% sensitivity; fine-tuned 84.76% / 83.04%).
5. **Efficiency:** relative to baseline VGG16T, ResNet50 has ≈5.6× fewer parameters (24.1 M vs 134.3 M), a ≈14.7× smaller checkpoint (96.6 MB vs 1,424.6 MB) and ≈2.7× higher CPU throughput (7.74 vs 2.88 images/s on the benchmark machine). The fine-tuned VGG16T checkpoint is larger (1,478.5 MB) and slower on CPU (1.56 images/s) because more of its weights are trainable and stored with optimizer state.
6. **Grad-CAM** maps for six test cases (4 correct, 1 false positive, 1 false negative) are provided for qualitative inspection only.

**Limitations**
- One official split and one training run (single seed) per model; seed-to-seed variability is not measured.
- ROI crops use the **expert-annotated bounding boxes at test time as well**, so the results assume a localized nodule and do not measure end-to-end detection + classification.
- The test set is 73% malignant (731/1,000), so accuracy is influenced by prevalence; sensitivity and specificity should be read together.
- TN5000 is a different dataset from the Zhu et al. cohort, so the 86.43% figure in the reference paper is not directly comparable; no external validation set was used.
- The efficiency timings are from a single benchmark run on one machine.